In [0]:
%python
# Databricks notebook source
from pyspark.sql import functions as F

# Leitura da tabela crua da camada Bronze
df_bronze = spark.read.table("market_prices_db.bronze.raw_food_prices")

# Limpeza e Padronização de Colunas sem perda de registros e mapeando colunas reais
df_silver = (
    df_bronze.withColumn("adm0_id", F.col("adm0_id").cast("integer"))
    .withColumn("adm0_name", F.trim(F.col("adm0_name")))
    .withColumn("adm1_id", F.col("adm1_id").cast("integer"))
    .withColumn("adm1_name", F.trim(F.col("adm1_name")))
    .withColumn("mkt_id", F.col("mkt_id").cast("integer"))
    .withColumn("mkt_name", F.trim(F.col("mkt_name")))
    .withColumn("cm_id", F.col("cm_id").cast("integer"))
    .withColumn("cm_name", F.trim(F.col("cm_name")))
    .withColumn("cur_id", F.col("cur_id").cast("integer"))
    .withColumn("cur_name", F.trim(F.col("cur_name")))
    .withColumn("pt_id", F.col("pt_id").cast("integer"))
    .withColumn("pt_name", F.trim(F.col("pt_name")))
    .withColumn("um_id", F.col("um_id").cast("integer"))
    .withColumn("um_name", F.trim(F.col("um_name")))
    # Tratamento seguro para preços e conversões numéricas
    .withColumn("price", F.col("mp_price").cast("double"))
    .withColumn("year", F.col("mp_year").cast("integer"))
    .withColumn("month", F.col("mp_month").cast("integer"))
    .withColumn(
        "commodity_source", F.trim(F.col("mp_commoditysource"))
    )
    # Geração segura da data de referência (Ano-Mês-01) de forma condicional
    .withColumn(
        "reference_date",
        F.when(
            F.col("mp_year").isNotNull() & F.col("mp_month").isNotNull(),
            F.to_date(
                F.concat_ws(
                    "-",
                    F.col("mp_year"),
                    F.lpad(F.col("mp_month"), 2, "0"),
                    F.lit("01"),
                ),
                "yyyy-MM-dd",
            ),
        ).otherwise(
            F.lit(None)
        ),
    )
    # Remove apenas duplicatas exatas para preservar absolutamente todos os dados
    .dropDuplicates()
)

# Persistência na camada Silver usando Delta Lake
target_table = "market_prices_db.silver.clean_food_prices"
df_silver.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(target_table)

total_registros = df_silver.count()
print(
    f"Camada Silver atualizada com sucesso sem perda de registros. Total de"
    f" linhas na Silver: {total_registros}"
)